# 02 - Synthetic Corpus Generation

This notebook generates synthetic parallel data using Gemma 31B through back-translation and paraphrase augmentation.

**Techniques:**
- Back-translation through English pivot
- Paraphrase augmentation
- Domain-specific dialogue synthesis

## Setup

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
import os
import sys
from pathlib import Path
import json
import torch
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data.corpus_generator import CorpusGenerator, GemmaTranslationModel, SyntheticPair

SEED_DIR = Path('../data/seed')
OUTPUT_DIR = Path('../data/synthetic')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load Teacher Model

In [ ]:
# Configuration
TEACHER_MODEL = "google/gemma-2-27b-it"  # Or use kaggle model path
SOURCE_LANG = "so"
TARGET_LANG = "sw"
PIVOT_LANG = "en"

# Initialize model (will load lazily)
model = GemmaTranslationModel(
    model_name=TEACHER_MODEL,
    device="cuda",
    temperature=0.7,
    top_p=0.9
)

In [ ]:
# Initialize corpus generator
generator = CorpusGenerator(
    model=model,
    source_lang=SOURCE_LANG,
    target_lang=TARGET_LANG,
    pivot_lang=PIVOT_LANG,
    output_dir=str(OUTPUT_DIR / f"{SOURCE_LANG}-{TARGET_LANG}")
)

## Load Seed Data

In [ ]:
# Load source sentences for back-translation
seed_path = SEED_DIR / f"{SOURCE_LANG}-{TARGET_LANG}" / f"seed.{SOURCE_LANG}"

with open(seed_path, 'r', encoding='utf-8') as f:
    source_sentences = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(source_sentences)} source sentences")

## Generate Back-Translation Pairs

In [ ]:
# Generate synthetic pairs via back-translation
MAX_PAIRS = 10000  # Adjust based on GPU quota

bt_pairs = []
for pair in tqdm(generator.generate_back_translation_pairs(
    source_sentences[:MAX_PAIRS],
    domain="humanitarian"
), total=min(MAX_PAIRS, len(source_sentences))):
    bt_pairs.append(pair)

print(f"Generated {len(bt_pairs)} back-translation pairs")

## Generate Domain Dialogues

In [ ]:
# Generate domain-specific dialogues
medical_dialogues = list(generator.generate_domain_dialogue(
    domain="medical",
    num_dialogues=500
))

legal_dialogues = list(generator.generate_domain_dialogue(
    domain="legal",
    num_dialogues=500
))

humanitarian_dialogues = list(generator.generate_domain_dialogue(
    domain="humanitarian",
    num_dialogues=500
))

all_dialogues = medical_dialogues + legal_dialogues + humanitarian_dialogues
print(f"Generated {len(all_dialogues)} domain dialogue pairs")

## Save Generated Data

In [ ]:
# Combine all synthetic pairs
all_pairs = bt_pairs + all_dialogues

# Save corpus
generator.save_corpus(all_pairs, prefix="synthetic")
generator.save_generation_log()

In [ ]:
print("=" * 50)
print("SYNTHETIC CORPUS GENERATION COMPLETE")
print("=" * 50)
print(f"\nTotal pairs generated: {len(all_pairs):,}")
print(f"  - Back-translation: {len(bt_pairs):,}")
print(f"  - Domain dialogues: {len(all_dialogues):,}")
print(f"\nNext step: Run 03_corpus_validation.ipynb")